In [ ]:
import numpy as np
import pandas as pd
import pyvista as pv
import sdf

pv.set_jupyter_backend("trame")

### Trident

In [ ]:
waypoints = {
  "main": np.array(
    [
      [0.000, 0.000, 0.000],
      [0.040, 0.000, 0.000],
      [0.070, 0.000, 0.000],
      [0.120, 0.000, 0.000],
      [0.165, 0.000, 0.000],
      [0.210, 0.000, 0.000],
    ]
  ),
  "upper": np.array(
    [
      [0.070, 0.000, 0.000],
      [0.095, 0.020, 0.010],
      [0.130, 0.030, 0.024],
      [0.170, 0.032, 0.026],
      [0.200, 0.032, 0.026],
    ]
  ),
  "lower": np.array(
    [
      [0.070, 0.000, 0.000],
      [0.098, -0.024, -0.008],
      [0.130, -0.032, -0.012],
      [0.152, -0.026, -0.008],
      [0.165, 0.000, 0.000],
    ]
  ),
}
bore_radius = 5.0e-3
wall_thickness = 2.0e-3
cut_x = 0.004

centerlines = {
  name: np.asarray(pv.Spline(points, 40).points) for name, points in waypoints.items()
}
all_points = np.concatenate(list(centerlines.values()))
padding = bore_radius + wall_thickness + 0.003
bounds = (
  tuple(all_points.min(axis=0) - padding),
  tuple(all_points.max(axis=0) + padding),
)

capsules = [
  sdf.capsule(start, end, bore_radius + 0.5 * wall_thickness)
  for line in centerlines.values()
  for start, end in zip(line[:-1], line[1:])
]


In [ ]:

network = capsules[0].union(*capsules[1:]).shell(wall_thickness) & sdf.slab(x0=cut_x)
network.save(
  "crops/data/obstacles/network.stl",
  bounds=bounds,
  step=wall_thickness / 4.0,
)
print(f"{len(capsules)} capsules, bounds {np.round(bounds, 3)}")

### Helix

In [ ]:
case_name = "helix_a3_l20"
is_helix = True  # False gives the planar sine
lumen_radius = 1.5e-3
wall_thickness = 2.0e-3  # thick wall keeps the mid-wall SDF kink far from the lumen
lead_in_length = 10.0e-3
amplitude = 3.0e-3  # tortuosity knob, helix radius when is_helix
wavelength = 40.0e-3  # pitch; p = 6.4 mm sits outside the clearance band
num_periods = 3
points_per_period = 64 

num_points = points_per_period * num_periods + 1
axial = np.linspace(0.0, num_periods * wavelength, num_points)
phase = 2.0 * np.pi * axial / wavelength
lateral = amplitude * (1.0 - np.cos(phase))
out_of_plane = is_helix * amplitude * np.sin(phase)
curve = np.stack([axial, lateral, out_of_plane], axis=-1)

# The lead-in follows the start tangent, since a helix never leaves along its axis.
start_step = curve[1] - curve[0]
start_tangent = start_step / np.linalg.norm(start_step)
end_step = curve[-1] - curve[-2]
end_tangent = end_step / np.linalg.norm(end_step)
mouth = curve[0] - lead_in_length * start_tangent

lead_in_arc = np.linspace(0.0, lead_in_length, 10, endpoint=False)
lead_in = mouth + np.einsum("n,d->nd", lead_in_arc, start_tangent)
centerline = np.concatenate([lead_in, curve], axis=0)

In [ ]:
capsule_radius = lumen_radius + 0.5 * wall_thickness
capsules = [
  sdf.capsule(start, end, capsule_radius)
  for start, end in zip(centerline[:-1], centerline[1:])
]
# Half-spaces that keep the tube between the mouth and the far end.
mouth_cut = sdf.plane(normal=start_tangent, point=mouth)
far_cut = sdf.plane(normal=-end_tangent, point=curve[-1])
tube = capsules[0].union(*capsules[1:]).shell(wall_thickness) & mouth_cut & far_cut

padding = capsule_radius + wall_thickness
lower_corner = centerline.min(axis=0) - padding
upper_corner = centerline.max(axis=0) + padding
bounds = (tuple(lower_corner), tuple(upper_corner))
step = lumen_radius / 10.0
tube.save(f"crops/data/obstacles/{case_name}.stl", bounds=bounds, step=step)

segments = np.diff(centerline, axis=0)
segment_lengths = np.linalg.norm(segments, axis=-1)
cumulative = np.cumsum(segment_lengths)
arc_coords = np.concatenate([[0.0], cumulative])
columns = {
  "s": arc_coords,
  "x": centerline[:, 0],
  "y": centerline[:, 1],
  "z": centerline[:, 2],
}
pd.DataFrame(columns).to_csv(f"crops/data/targets/{case_name}.csv", index=False)